## Imports

In [28]:
import sys
import os
import torch
import torch.utils.data as data
import torch.nn as nn

sys.path.append(os.path.abspath('../../'))

import neuro_fuzzy_toolbox as nft

In [29]:
import numpy as np

In [30]:
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

from sklearn.metrics import (
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    accuracy_score,
    classification_report,
    multilabel_confusion_matrix
)

In [31]:
from ucimlrepo import fetch_ucirepo

# Parkinsons

## Data

In [32]:
parkinsons = fetch_ucirepo(id=174)

X = parkinsons.data.features
y = parkinsons.data.targets

In [33]:
parkinsons.variables

,name,role,type,demographic,description,units,missing_values
0,name,ID,Categorical,None,None,None,no
1,MDVP:Fo,Feature,Continuous,None,None,Hz,no
2,MDVP:Fhi,Feature,Continuous,None,None,Hz,no
3,MDVP:Flo,Feature,Continuous,None,None,Hz,no
4,MDVP:Jitter,Feature,Continuous,None,None,%,no
5,MDVP:Jitter,Feature,Continuous,None,None,Abs,no
6,MDVP:RAP,Feature,Continuous,None,None,None,no
7,MDVP:PPQ,Feature,Continuous,None,None,None,no
8,Jitter:DDP,Feature,Continuous,None,None,None,no
9,MDVP:Shimmer,Feature,Continuous,None,None,None,no


In [34]:
X

,MDVP:Fo,MDVP:Fhi,MDVP:Flo,MDVP:Jitter,MDVP:Jitter,MDVP:RAP,MDVP:PPQ,Jitter:DDP,MDVP:Shimmer,MDVP:Shimmer,...,MDVP:APQ,Shimmer:DDA,NHR,HNR,RPDE,DFA,spread1,spread2,D2,PPE
0,119.992,157.302,74.997,0.00784,0.00784,0.00370,0.00554,0.01109,0.04374,0.04374,...,0.02971,0.06545,0.02211,21.033,0.414783,0.815285,-4.813031,0.266482,2.301442,0.284654
1,122.400,148.650,113.819,0.00968,0.00968,0.00465,0.00696,0.01394,0.06134,0.06134,...,0.04368,0.09403,0.01929,19.085,0.458359,0.819521,-4.075192,0.335590,2.486855,0.368674
2,116.682,131.111,111.555,0.01050,0.01050,0.00544,0.00781,0.01633,0.05233,0.05233,...,0.03590,0.08270,0.01309,20.651,0.429895,0.825288,-4.443179,0.311173,2.342259,0.332634
3,116.676,137.871,111.366,0.00997,0.00997,0.00502,0.00698,0.01505,0.05492,0.05492,...,0.03772,0.08771,0.01353,20.644,0.434969,0.819235,-4.117501,0.334147,2.405554,0.368975
4,116.014,141.781,110.655,0.01284,0.01284,0.00655,0.00908,0.01966,0.06425,0.06425,...,0.04465,0.10470,0.01767,19.649,0.417356,0.823484,-3.747787,0.234513,2.332180,0.410335
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
190,174.188,230.978,94.261,0.00459,0.00459,0.00263,0.00259,0.00790,0.04087,0.04087,...,0.02745,0.07008,0.02764,19.517,0.448439,0.657899,-6.538586,0.121952,2.657476,0.133050
191,209.516,253.017,89.488,0.00564,0.00564,0.00331,0.00292,0.00994,0.02751,0.02751,...,0.01879,0.04812,0.01810,19.147,0.431674,0.683244,-6.195325,0.129303,2.784312,0.168895
192,174.688,240.005,74.287,0.01360,0.01360,0.00624,0.00564,0.01873,0.02308,0.02308,...,0.01667,0.03804,0.10715,17.883,0.407567,0.655683,-6.787197,0.158453,2.679772,0.131728
193,198.764,396.961,74.904,0.00740,0.00740,0.00370,0.00390,0.01109,0.02296,0.02296,...,0.01588,0.03794,0.07223,19.020,0.451221,0.643956,-6.744577,0.207454,2.138608,0.123306


In [35]:
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y)
x_train, x_val, y_train, y_val = train_test_split(x_train, y_train, test_size=0.2, stratify=y_train)

In [36]:
unique_classes, counts = np.unique(y_train.values, return_counts=True)
print("Training Data")
print(unique_classes)
print(counts)

Training Data
[0 1]
[26 82]


In [37]:
unique_classes, counts = np.unique(y_val.values, return_counts=True)
print("Validation Data")
print(unique_classes)
print(counts)

Validation Data
[0 1]
[ 7 21]


In [38]:
unique_classes, counts = np.unique(y_test.values, return_counts=True)
print("Testing Data")
print(unique_classes)
print(counts)

Testing Data
[0 1]
[15 44]


In [39]:
scaler = MinMaxScaler(feature_range=(-1, 1))

x_train = scaler.fit_transform(x_train)

x_val = scaler.transform(x_val)
x_test = scaler.transform(x_test)

In [40]:
x_train = torch.from_numpy(x_train)
x_val = torch.from_numpy(x_val)
x_test = torch.from_numpy(x_test)

y_train = torch.from_numpy(y_train.values).squeeze()
y_val = torch.from_numpy(y_val.values).squeeze()
y_test = torch.from_numpy(y_test.values).squeeze()

## Model & Algorithms

### DataLoaders

In [41]:
train_loader = data.DataLoader(
    data.TensorDataset(x_train, y_train), 
    batch_size = 8, 
    shuffle = True)
x_train = train_loader.dataset.tensors[0]
y_train = train_loader.dataset.tensors[1]

In [42]:
val_loader = data.DataLoader(
    data.TensorDataset(x_val, y_val), 
    batch_size = 8, 
    shuffle = True)
x_val = val_loader.dataset.tensors[0]
y_val = val_loader.dataset.tensors[1]

### ANFIS

In [43]:
model = nft.rule_reduced_ANFIS(
    input_size = x_train.shape[1],
    num_mfs = 1,
    outputs = 2,
    output_type='softmax',
    dtype=torch.float64
)

model.init_premises(x_train)

In [44]:
with torch.no_grad():
    pred = model(x_train)

nn.functional.cross_entropy(pred, y_train)

tensor(0.7668, dtype=torch.float64)

In [45]:
#model.init_consequents(x_train, y_train, driver="gelsd", ridge_lambda=1.0)
"""
with torch.no_grad():
    pred = model(x_train)

nn.functional.cross_entropy(pred, y_train)
"""

'\nwith torch.no_grad():\n    pred = model(x_train)\n\nnn.functional.cross_entropy(pred, y_train)\n'

### Learning Algorithm

In [46]:
#loss_fn = nn.functional.binary_cross_entropy
loss_fn = nn.functional.cross_entropy
epochs = 1000
optimizer = torch.optim.Adam
params = {'lr': 0.001}
#optimizer = torch.optim.AdamW
#params = {'lr': 0.01, 'weight_decay': 0.01}
#optimizer = torch.optim.SGD
#params = {'lr': 0.001, 'momentum': 0.9}

early_stopping = nft.EarlyStopping(
    patience=30, 
    #delta=0.0001
)

In [47]:
trainer = nft.Hybrid_learning_algorithm(
    epochs=epochs,
    loss_function=loss_fn,
    ridge_lambda=1.,
    optimizer=optimizer,
    optimizer_params=params,
    early_stopping=early_stopping
)

In [48]:
"""
trainer(model, train_loader, val_loader, verbose=True)

pred = model.predict(x_test)

acc = accuracy_score(y_test, pred)
prec = precision_score(y_test, pred, zero_division=0)
recall = recall_score(y_test, pred)
f1 = f1_score(y_test, pred, zero_division=0)
conf_matrix = confusion_matrix(y_test, pred)
class_rep = classification_report(y_test, pred)

print("\n ------ Evaluation ------")
print("\n")
print("Accuracy:", acc)
print("Precision:", prec)
print("Recall:", recall)
print("f1 score:", f1, "\n")

print("Confusion Matrix:")
print(conf_matrix, "\n")

print("Classification Report:")
print(class_rep)
"""

'\ntrainer(model, train_loader, val_loader, verbose=True)\n\npred = model.predict(x_test)\n\nacc = accuracy_score(y_test, pred)\nprec = precision_score(y_test, pred, zero_division=0)\nrecall = recall_score(y_test, pred)\nf1 = f1_score(y_test, pred, zero_division=0)\nconf_matrix = confusion_matrix(y_test, pred)\nclass_rep = classification_report(y_test, pred)\n\nprint("\n ------ Evaluation ------")\nprint("\n")\nprint("Accuracy:", acc)\nprint("Precision:", prec)\nprint("Recall:", recall)\nprint("f1 score:", f1, "\n")\n\nprint("Confusion Matrix:")\nprint(conf_matrix, "\n")\n\nprint("Classification Report:")\nprint(class_rep)\n'

### SONFIS

In [49]:
Ngrow = 20
dGrow = 0.8
Nsplit = 25
eSplit = 0.45
Nvanish = 5
lVanish = 2

max_iterations = 100

anfis_trainer = trainer

sonfis_early_stopping = nft.EarlyStopping(patience=20, delta=0.01)
last_training_iteration = False

In [50]:
sonfis = nft.SONFIS(
    Ngrow=Ngrow,
    dGrow=dGrow,
    Nsplit=Nsplit,
    eSplit=eSplit,
    Nvanish=Nvanish,
    lVanish=lVanish,
    max_iterations=max_iterations,
    ANFIStrainer=anfis_trainer,
    early_stopping=sonfis_early_stopping,
    last_training_iteration=last_training_iteration
)

## Training

In [51]:
%time sonfis(model, train_loader, val_loader, verbose=True)

ITERATION:   0/100



STARTING STATE:
   premises                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                      output 1 consequents                                                                                                                                                                                                                             output 2 consequents                                                                                                                                                                                 

## Evaluation

### Test data

In [52]:
pred = model.predict(x_test)

acc = accuracy_score(y_test, pred)
prec = precision_score(y_test, pred, zero_division=0)
recall = recall_score(y_test, pred)
f1 = f1_score(y_test, pred, zero_division=0)
conf_matrix = confusion_matrix(y_test, pred)
#mul_label_conf_matrix = multilabel_confusion_matrix(y_test, pred)
class_rep = classification_report(y_test, pred)

print("Accuracy:", acc)
print("Precision:", prec)
print("Recall:", recall)
print("f1 score:", f1, "\n")

print("Confusion Matrix:")
print(conf_matrix, "\n")

#print("Multilabel Confusion Matrix:")
#print(mul_label_conf_matrix, "\n")

print("Classification Report:")
print(class_rep)

Accuracy: 0.8813559322033898
Precision: 0.9302325581395349
Recall: 0.9090909090909091
f1 score: 0.9195402298850575 

Confusion Matrix:
[[12  3]
 [ 4 40]] 

Classification Report:
              precision    recall  f1-score   support

           0       0.75      0.80      0.77        15
           1       0.93      0.91      0.92        44

    accuracy                           0.88        59
   macro avg       0.84      0.85      0.85        59
weighted avg       0.88      0.88      0.88        59



### Training data

In [53]:
pred = model.predict(x_train)

acc = accuracy_score(y_train, pred)
prec = precision_score(y_train, pred, zero_division=0)
recall = recall_score(y_train, pred)
f1 = f1_score(y_train, pred, zero_division=0)
conf_matrix = confusion_matrix(y_train, pred)
class_rep = classification_report(y_train, pred)

print("Accuracy:", acc)
print("Precision:", prec)
print("Recall:", recall)
print("f1 score:", f1, "\n")

print("Confusion Matrix:")
print(conf_matrix, "\n")

print("Classification Report:")
print(class_rep)

Accuracy: 0.9907407407407407
Precision: 1.0
Recall: 0.9878048780487805
f1 score: 0.9938650306748467 

Confusion Matrix:
[[26  0]
 [ 1 81]] 

Classification Report:
              precision    recall  f1-score   support

           0       0.96      1.00      0.98        26
           1       1.00      0.99      0.99        82

    accuracy                           0.99       108
   macro avg       0.98      0.99      0.99       108
weighted avg       0.99      0.99      0.99       108



In [54]:
print(model.get_rules_structure().to_string())

        premises                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                           output 1 consequents                                                                                                                                                                                                                             output 2 consequents                                                                        